# MegaDescriptor 파인튜닝 — MPDD  (wildlife-tools 공식 레시피)

공식 학습 노트북 `wildlife-tools/baselines/training/MegaDescriptor-T-224.ipynb` 을 MPDD 용으로 옮긴 것.

**공식과 동일하게 유지한 부분**
- `wildlife_tools.data.WildlifeDataset` + `wildlife_tools.train.ArcFaceLoss` + `BasicTrainer`
- ArcFace `margin=0.5`, `scale=64`
- 증강: `RandomResizedCrop` + `RandAugment(2, 20)`
- 옵티마이저: `SGD(lr=1e-3, momentum=0.9)` + `CosineAnnealingLR`
- `batch_size=64`, `accumulation_steps=2` (유효 배치 128)

**MPDD 용으로 바꾼 부분**
- 데이터: MPDD (Market-1501 스타일) → 메타데이터 DataFrame 직접 구성
- 백본: ImageNet Swin 대신 **기존 MegaDescriptor 가중치에서 이어서** 파인튜닝
- `epochs` 100 → 30 (개체 95개뿐이라 그 이상은 과적합)
- `epoch_callback` 으로 매 epoch MPDD query/gallery 평가 + best 저장

> GPU 필요. CPU면 아주 느립니다 (Colab T4 권장).


## 0. 설치

In [1]:
!pip install -q wildlife-tools timm

## 1. 임포트 & 설정

In [ ]:
import os, re
from itertools import chain

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.optim import SGD
from torch.utils.data import DataLoader

import timm
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.train import ArcFaceLoss, BasicTrainer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- 1. MPDD_ROOT 경로 ----
MPDD_ROOT = "./dataset/Multi-pose dog dataset/MPDD/pytorch"          # 안에 train/ val/ gallery/ query/
# ---- 2. 보호소 동물 경로 ----
SHELTER_ROOT = "processed_animals"  # 개체당 폴더 (최종 평가용). 없으면 None

# ---- 하이퍼파라미터 (공식값 + MPDD 조정) ----
MODEL = "megadescriptor"     # "megadescriptor" | "petface"

if MODEL == "megadescriptor":
    BACKBONE     = "hf-hub:BVRA/MegaDescriptor-B-224"
    CKPT         = "megadescriptor_mpdd_best.pth"
    LR           = 1e-3 # Learning Rate: 학습할 때 가중치를 얼마나 크게 변경할지 결정
elif MODEL == "petface":
    PETFACE_CKPT = "petface_pretrained/dog.pt"
    CKPT         = "petface_mpdd_best.pth"
    LR           = 1e-4 

EPOCHS    = 30       # 학습데이터 EPOCHS번 반복해서 학습
BATCH     = 64       # 한 번에 GPU에 넣는 이미지 수
ACCUM     = 2        # gradient update는 BATCH * ACCUM장을 본 뒤 한 번 한다.
IMG_SIZE  = 224      # 모델에 넣을 이미지 크기
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225) # 정규화할 때 사용할 값

# train 폴더가 진짜 존재하는지 검사
if not os.path.isdir(f"{MPDD_ROOT}/train"):
    raise AssertionError(f"train 폴더 없음: {MPDD_ROOT}/train")
print("DEVICE:", DEVICE, "| MODEL:", MODEL)


c:\HyeonKyu\for_mate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda | BACKBONE: hf-hub:BVRA/MegaDescriptor-B-224


## 2. MPDD 메타데이터 구성

Market-1501 파일명 `<id>_c<pose>s<seq>_<n>.jpg` 에서 identity / camera 를 뽑아 DataFrame 으로.

In [2]:
_ID = re.compile(r"^(\d+)_c(\d+)s\d+")

def market_meta(split):
    d = f"{MPDD_ROOT}/{split}" # ex) {MPDD_ROOT}/train
    rows = []
    for f in sorted(os.listdir(d)):
        m = _ID.match(f) # 앞서 만든 정규표현식의 규칙과 매칭되는지 확인
        if m and f.lower().endswith((".jpg", ".jpeg", ".png")): # 1) 원하는 패턴인지? 2) 이미지 파일인지 확인
            rows.append({"path": f"{split}/{f}", "identity": m.group(1),
                         "camera": int(m.group(2)), "split": split})
    return pd.DataFrame(rows)

meta_train = market_meta("train")   # 파인튜닝에 사용할 사진
meta_gal   = market_meta("gallery") # 기준으로 등록해놓은 사진
meta_qry   = market_meta("query")   # 누구인지 찾아야 하는 사진

print(f"train   {len(meta_train):4d}장 / {meta_train['identity'].nunique()}개체")
print(f"gallery {len(meta_gal):4d}장 / {meta_gal['identity'].nunique()}개체")
print(f"query   {len(meta_qry):4d}장 / {meta_qry['identity'].nunique()}개체")
print("train ∩ gallery 개체:",
      len(set(meta_train.identity) & set(meta_gal.identity)), "(0 = open-set, 정상)")


train    921장 / 95개체
gallery  521장 / 96개체
query    103장 / 95개체
train ∩ gallery 개체: 0 (0 = open-set, 정상)


## 3. 데이터셋 & 증강  *(공식과 동일)*

In [3]:
train_tf = T.Compose([
    T.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0)), # 원본 면적의 80~100% 정도를 사용하는 범위에서 crop을 선택
    T.RandAugment(num_ops=2, magnitude=20), # 여러 이미지 증강 방법 중 랜덤하게 몇 개를 선택해서 적용
    T.ToTensor(), # 이미지를 PyTorch Tensor로 변환
    T.Normalize(mean=MEAN, std=STD), # 각 RGB 채널 정규화
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

train_ds = WildlifeDataset(metadata=meta_train, root=MPDD_ROOT, transform=train_tf)
print("num_classes:", train_ds.num_classes)


num_classes: 95


## 4. 백본 & ArcFace  *(공식과 동일)*

In [ ]:
if MODEL == "megadescriptor":
    backbone = timm.create_model(BACKBONE, num_classes=0, pretrained=True)
elif MODEL == "petface":
    from torchvision.models import resnet50
    import torch.nn as nn
    backbone = resnet50(weights=None)
    backbone.fc = nn.Sequential(nn.Linear(backbone.fc.in_features, 512), nn.BatchNorm1d(512))
    sd = torch.load(PETFACE_CKPT, map_location="cpu", weights_only=False)["state_dict_backbone"]
    backbone.load_state_dict(sd)

with torch.no_grad():
    embedding_size = backbone(torch.randn(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]

objective = ArcFaceLoss(
    num_classes=train_ds.num_classes,
    embedding_size=embedding_size,
    margin=0.5,
    scale=64,
)
print("embedding_size:", embedding_size)


embedding_size: 1024


## 5. 옵티마이저 & 스케줄러  *(공식과 동일, T_max만 EPOCHS)*

In [5]:
params = chain(backbone.parameters(), objective.parameters())
optimizer = SGD(params=params, lr=LR, momentum=0.9)

# val mAP 가 patience 에폭 동안 안 오르면 LR 을 factor 배로
plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=LR * 1e-3,
)

PATIENCE = 5   # 연속 이만큼 개선 없으면 early stop

## 6. 평가 함수 — MPDD query / gallery

re-ID 표준: 각 query 를 gallery 와 코사인 유사도로 랭킹.
Market 규칙대로 **같은 개체 & 같은 pose** 는 junk 로 제외. → mAP, Top-1, Top-5.

In [6]:
@torch.no_grad()
def embed(model, meta, root):
    ds = WildlifeDataset(metadata=meta, root=root, transform=eval_tf, load_label=False)
    dl = DataLoader(ds, batch_size=128, num_workers=0)
    model.eval()
    out = [F.normalize(model(x.to(DEVICE))).cpu() for x in dl]
    return torch.cat(out).numpy()


def _ap_cmc(rel):
    if not rel.any():
        return 0.0, 0, 0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())


def evaluate_qg(q_emb, q_id, q_cam, g_emb, g_id, g_cam):
    q_id, g_id = np.asarray(q_id), np.asarray(g_id)
    q_cam, g_cam = np.asarray(q_cam), np.asarray(g_cam)
    sim = q_emb @ g_emb.T
    aps = c1 = c5 = 0.0
    for i in range(len(q_id)):
        order = np.argsort(sim[i])[::-1]
        junk = (g_id == q_id[i]) & (g_cam == q_cam[i])
        order = order[~junk[order]]
        ap, h1, h5 = _ap_cmc(g_id[order] == q_id[i])
        aps += ap; c1 += h1; c5 += h5
    n = len(q_id)
    return aps / n, c1 / n, c5 / n


def mpdd_score(model):
    qe = embed(model, meta_qry, MPDD_ROOT)
    ge = embed(model, meta_gal, MPDD_ROOT)
    return evaluate_qg(qe, meta_qry.identity, meta_qry.camera,
                       ge, meta_gal.identity, meta_gal.camera)


## 7. epoch 콜백 — 매 epoch MPDD 평가 + best 저장

In [7]:
best = {"mAP": -1.0, "epoch": -1}
_bad = {"n": 0}

class EarlyStop(Exception):
    pass

def on_epoch(trainer, epoch_data):
    mAP, t1, t5 = mpdd_score(trainer.model)
    lr = optimizer.param_groups[0]["lr"]
    print(f"  [epoch {trainer.epoch}] loss {epoch_data['train_loss_epoch_avg']:.3f}"
          f"  | lr {lr:.2e} | MPDD  mAP {mAP:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")

    if mAP > best["mAP"] + 1e-4:                 # 유의미 개선일 때만
        best.update(mAP=mAP, epoch=trainer.epoch)
        _bad["n"] = 0
        trainer.save(".", CKPT)
        print(f"      -> best 저장 (mAP {mAP:.4f})  {CKPT}")
    else:
        _bad["n"] += 1
        print(f"      개선 없음 {_bad['n']}/{PATIENCE}")

    plateau.step(mAP)                            # 점수 정체 시 LR 컷

    if _bad["n"] >= PATIENCE:
        print(f"  early stop @ epoch {trainer.epoch} — best = epoch {best['epoch']} (mAP {best['mAP']:.4f})")
        raise EarlyStop

## 8. 학습  *(BasicTrainer — 공식과 동일 구조)*

In [8]:
trainer = BasicTrainer(
    dataset=train_ds,
    model=backbone,
    objective=objective,
    optimizer=optimizer,
    scheduler=None,                # 스케줄러는 on_epoch 에서 plateau 로 수동 처리
    batch_size=BATCH,
    accumulation_steps=ACCUM,
    num_workers=0,
    epochs=EPOCHS,
    device=DEVICE,
    epoch_callback=on_epoch,
)

print("zero-shot(학습 전):  MPDD  mAP {:.4f}  Top-1 {:.4f}  Top-5 {:.4f}".format(*mpdd_score(backbone)))
try:
    trainer.train()
except EarlyStop:
    pass
print(f"\n최고 MPDD mAP: {best['mAP']:.4f} @ epoch {best['epoch']}  (저장: {CKPT})")

zero-shot(학습 전):  MPDD  mAP 0.6939  Top-1 0.8447  Top-5 0.9417


Epoch 0: 100%|██████████████████████████████████████████████████████| 15/15 [01:27<00:00,  5.84s/it]


  [epoch 1] loss 36.532  | lr 1.00e-03 | MPDD  mAP 0.6792  Top-1 0.8155  Top-5 0.9515
      -> best 저장 (mAP 0.6792)  megadescriptor_mpdd_best.pth


Epoch 1: 100%|██████████████████████████████████████████████████████| 15/15 [07:20<00:00, 29.35s/it]


  [epoch 2] loss 33.886  | lr 1.00e-03 | MPDD  mAP 0.6925  Top-1 0.8641  Top-5 0.9612
      -> best 저장 (mAP 0.6925)  megadescriptor_mpdd_best.pth


...

  [epoch 21] loss 3.542  | lr 5.00e-04 | MPDD  mAP 0.8055  Top-1 0.9417  Top-5 0.9903
      개선 없음 3/5


Epoch 21: 100%|█████████████████████████████████████████████████████| 15/15 [06:58<00:00, 27.91s/it]


  [epoch 22] loss 3.429  | lr 2.50e-04 | MPDD  mAP 0.8020  Top-1 0.9320  Top-5 0.9903
      개선 없음 4/5


Epoch 22: 100%|█████████████████████████████████████████████████████| 15/15 [06:58<00:00, 27.87s/it]


  [epoch 23] loss 3.471  | lr 2.50e-04 | MPDD  mAP 0.8049  Top-1 0.9515  Top-5 0.9903
      개선 없음 5/5
  early stop @ epoch 23 — best = epoch 18 (mAP 0.8117)

최고 MPDD mAP: 0.8117 @ epoch 18  (저장: megadescriptor_mpdd_best.pth)


## 9. 최종 평가 — 보호소 데이터(`processed_animals`)

MPDD 숫자는 논문 비교용. **실제 제품 지표는 여기.**
best 체크포인트를 백본에 로드해 개체당 절반 gallery / 절반 query 로 mAP·Top-1 (5시드 평균).

In [11]:
import os, numpy as np, torch, torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
SHELTER_ROOT = "processed_animals"
CKPT  = "megadescriptor_mpdd_best.pth"
MODEL = "megadescriptor"        # 학습한 것과 일치

# 1. 아키텍처 재구성
if MODEL == "megadescriptor":
    import timm
    model = timm.create_model("hf-hub:BVRA/MegaDescriptor-B-224", num_classes=0, pretrained=False)
elif MODEL == "petface":
    from torchvision.models import resnet50
    import torch.nn as nn
    model = resnet50(weights=None)
    model.fc = nn.Sequential(nn.Linear(model.fc.in_features, 512), nn.BatchNorm1d(512))

ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
model.eval().to(DEVICE)

tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def embed_folder(root):
    embs, ids = [], []
    for d in sorted(os.listdir(root)):
        dd = os.path.join(root, d)
        if not os.path.isdir(dd): continue
        for f in sorted(os.listdir(dd)):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                x = tf(Image.open(os.path.join(dd, f)).convert("RGB")).unsqueeze(0).to(DEVICE)
                embs.append(F.normalize(model(x)).cpu().numpy()[0]); ids.append(d)
    return np.array(embs), np.array(ids)

def _ap_cmc(rel):
    if not rel.any(): return 0.0, 0, 0
    hits = np.cumsum(rel); ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())

def eval_split(emb, ids, frac=0.5, seed=0):
    ids = np.asarray(ids); rng = np.random.default_rng(seed)
    g, q = [], []
    for c in np.unique(ids):
        idx = np.where(ids == c)[0].copy(); rng.shuffle(idx)
        if len(idx) <= 1: g += idx.tolist(); continue
        k = max(1, round(len(idx) * frac)); g += idx[:k].tolist(); q += idx[k:].tolist()
    g, q = np.array(g), np.array(q)
    sim = emb[q] @ emb[g].T; g_id = ids[g]
    ap = c1 = c5 = 0.0
    for row, gt in zip(sim, ids[q]):
        a, h1, h5 = _ap_cmc(g_id[np.argsort(row)[::-1]] == gt); ap += a; c1 += h1; c5 += h5
    n = len(q); return ap / n, c1 / n, c5 / n

emb, ids = embed_folder(SHELTER_ROOT)
r = np.array([eval_split(emb, ids, seed=s) for s in range(5)])
m, sd = r.mean(0), r.std(0)
print(f"개체 {len(set(ids))} / 이미지 {len(ids)}")
print(f"split  mAP {m[0]:.4f}±{sd[0]:.4f} | Top-1 {m[1]:.4f}±{sd[1]:.4f} | Top-5 {m[2]:.4f}±{sd[2]:.4f}")

KeyboardInterrupt: 

---
### 참고

- **전체 백본 파인튜닝**이 기본 (공식 `chain(backbone.parameters(), objective.parameters())`). 95개체로 과적합 조짐 보이면
  뒤쪽 stage만 학습하도록 셀 5 앞에 넣기:
  ```python
  for p in backbone.parameters(): p.requires_grad = False
  for p in backbone.layers[-1].parameters(): p.requires_grad = True
  for p in getattr(backbone, "norm", []).parameters(): p.requires_grad = True
  params = chain((p for p in backbone.parameters() if p.requires_grad), objective.parameters())
  ```
- 로컬 Windows/Jupyter 에서 돌리면 `num_workers=2` → `0` (셀 8, 셀 6).
- `BACKBONE` 을 `-L-384` 로 바꾸면 `BATCH=16`, `IMG_SIZE=384`.
